# Actualización periódica — Servicio de viento (ArcGIS Enterprise)

Este notebook está pensado para **programarse** (mensual, semanal, según
la frecuencia de tus fuentes) en ArcGIS Notebook Server, o ejecutarse vía
Task Scheduler si corres ArcGIS Pro localmente. Cada ejecución:

1. Vuelve a consultar los 3 Feature Services.
2. Genera un nuevo ráster `Vector-UV` con la fecha actual.
3. Lo agrega al Mosaic Dataset ya existente (creado en `01_setup_inicial_viento.ipynb`).
4. Opcionalmente elimina el ráster del período anterior.
5. Refresca el Image Service sin downtime (`RefreshService`, endpoint corregido
   respecto a la guía original).
6. Valida que el servicio quedó actualizado.

Requiere que `01_setup_inicial_viento.ipynb` ya se haya ejecutado y que el
Image Service ya esté publicado con `hasLiveData: true` activo.

In [ ]:
import arcpy
import numpy as np
import requests
from datetime import datetime
from scipy.interpolate import griddata

try:
    from arcpy.ia import VectorField
except ImportError:
    from arcpy.sa import VectorField

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURACION - debe coincidir con 01_setup_inicial_viento.ipynb
# ---------------------------------------------------------------------------

SERVICIOS = {
    "Ecuador":  "<FEATURE_SERVICE_URL_ECUADOR>",
    "Panama":   "<FEATURE_SERVICE_URL_PANAMA>",
    "Colombia": "<FEATURE_SERVICE_URL_COLOMBIA>",
}

CAMPO_LAT = "<CAMPO_LATITUD>"
CAMPO_LON = "<CAMPO_LONGITUD>"
CAMPO_VEL = "<CAMPO_VELOCIDAD_VIENTO>"
CAMPO_DIR = "<CAMPO_DIRECCION_VIENTO>"

CARPETA_REGISTRADA = r"<RUTA_CARPETA_REGISTRADA>"
GDB_NOMBRE = "VientoGDB"
MOSAICO_NOMBRE = "Viento_UV_Mosaico"
SPATIAL_REFERENCE = arcpy.SpatialReference(4326)

GDB_DESTINO = rf"{CARPETA_REGISTRADA}\{GDB_NOMBRE}.gdb"
MOSAICO_RUTA = rf"{GDB_DESTINO}\{MOSAICO_NOMBRE}"

# Datos del servicio ya publicado (paso 5 de 01_setup_inicial_viento.ipynb)
NOMBRE_SERVICIO = MOSAICO_NOMBRE   # por defecto igual al mosaic dataset; ajusta si le pusiste otro nombre al publicar
CARPETA_SERVICIO = "Viento-SoX"    # carpeta ya creada en ArcGIS Server Manager (Manage Services > Folders)
RUTA_HOST_AGS = r"<RUTA_ARCHIVO_CONEXION_AGS>"  # ej. r"C:\conexiones\host.ags" (conexion de publicador/admin)

# Inferido de tu ArcGIS Server Manager (arcgis.esri.co/image/manager) - confirma
# la ruta exacta una vez publicado el servicio. Patron REST estandar de un
# Web Adaptor: https://<host>/<nombre_web_adaptor>/rest/services/<carpeta>/<servicio>/ImageServer
IMAGE_SERVICE_URL = f"https://arcgis.esri.co/image/rest/services/{CARPETA_SERVICIO}/{NOMBRE_SERVICIO}/ImageServer"

# Si quieres eliminar el raster del periodo anterior automaticamente
ELIMINAR_RASTER_ANTERIOR = False
NOMBRE_RASTER_ANTERIOR = "<NOMBRE_RASTER_A_ELIMINAR>"  # ej. "viento_UV_202608"

## Paso 1 — Consultar los Feature Services

In [ ]:
def consultar_puntos(url_servicio):
    params = {
        "where": "1=1",
        "outFields": f"{CAMPO_LAT},{CAMPO_LON},{CAMPO_VEL},{CAMPO_DIR}",
        "returnGeometry": "false",
        "f": "json",
        "resultRecordCount": 2000,
    }
    r = requests.get(f"{url_servicio}/query", params=params, timeout=30)
    r.raise_for_status()
    out = []
    for feat in r.json().get("features", []):
        a = feat["attributes"]
        if None not in (a.get(CAMPO_LAT), a.get(CAMPO_LON), a.get(CAMPO_VEL), a.get(CAMPO_DIR)):
            out.append((a[CAMPO_LAT], a[CAMPO_LON], a[CAMPO_VEL], a[CAMPO_DIR]))
    return out


todos = []
for pais, url in SERVICIOS.items():
    puntos_pais = consultar_puntos(url)
    print(f"{pais}: {len(puntos_pais)} puntos")
    todos.extend(puntos_pais)

print(f"Total de puntos combinados: {len(todos)}")

lat_arr = np.array([p[0] for p in todos])
lon_arr = np.array([p[1] for p in todos])
vel_arr = np.array([p[2] for p in todos])
dir_arr = np.array([p[3] for p in todos])

## Paso 2 — Interpolar a grilla U/V

In [ ]:
dir_rad = np.deg2rad(dir_arr)
U = -vel_arr * np.sin(dir_rad)
V = -vel_arr * np.cos(dir_rad)

lon_min, lon_max = lon_arr.min() - 0.5, lon_arr.max() + 0.5
lat_min, lat_max = lat_arr.min() - 0.5, lat_arr.max() + 0.5
grid_lon, grid_lat = np.mgrid[lon_min:lon_max:300j, lat_min:lat_max:300j]

grid_U = griddata((lon_arr, lat_arr), U, (grid_lon, grid_lat), method="cubic")
grid_V = griddata((lon_arr, lat_arr), V, (grid_lon, grid_lat), method="cubic")

for g, fuente in [(grid_U, U), (grid_V, V)]:
    mask = np.isnan(g)
    if mask.any():
        g[mask] = griddata((lon_arr, lat_arr), fuente, (grid_lon[mask], grid_lat[mask]), method="nearest")

banda_U = np.flipud(grid_U.T).astype("float32")
banda_V = np.flipud(grid_V.T).astype("float32")

print(f"Grilla generada: {banda_U.shape}")

## Paso 3 — Generar el ráster del período y etiquetar `Vector-UV`

In [ ]:
punto_origen = arcpy.Point(lon_min, lat_min)
cell_size_x = (lon_max - lon_min) / banda_U.shape[1]
cell_size_y = (lat_max - lat_min) / banda_U.shape[0]

raster_U = arcpy.NumPyArrayToRaster(banda_U, punto_origen, cell_size_x, cell_size_y)
raster_V = arcpy.NumPyArrayToRaster(banda_V, punto_origen, cell_size_x, cell_size_y)

arcpy.management.DefineProjection(raster_U, SPATIAL_REFERENCE)
arcpy.management.DefineProjection(raster_V, SPATIAL_REFERENCE)

vector_field_raster = VectorField(raster_U, raster_V, "Vector-UV", "Geographic", "Vector-UV")

fecha_str = datetime.now().strftime("%Y%m")
nombre_raster_nuevo = f"viento_UV_{fecha_str}"
salida_periodo = rf"{CARPETA_REGISTRADA}\{nombre_raster_nuevo}.crf"

vector_field_raster.save(salida_periodo)
arcpy.management.DefineProjection(salida_periodo, SPATIAL_REFERENCE)

print(f"Raster del periodo generado: {salida_periodo}")

## Paso 4 — Agregar al Mosaic Dataset existente

In [ ]:
arcpy.management.AddRastersToMosaicDataset(
    in_mosaic_dataset=MOSAICO_RUTA,
    raster_type="Raster Dataset",
    input_path=salida_periodo,
)
print(f"Raster agregado al mosaic dataset: {nombre_raster_nuevo}")

## Paso 5 — (Opcional) eliminar el ráster del período anterior

Desactivado por defecto (`ELIMINAR_RASTER_ANTERIOR = False`) para
conservar histórico. Actívalo si prefieres que el mosaic dataset solo
contenga el período más reciente.

In [ ]:
if ELIMINAR_RASTER_ANTERIOR:
    arcpy.management.RemoveRastersFromMosaicDataset(
        MOSAICO_RUTA, where_clause=f"Name = '{NOMBRE_RASTER_ANTERIOR}'"
    )
    print(f"Raster anterior eliminado: {NOMBRE_RASTER_ANTERIOR}")
else:
    print("ELIMINAR_RASTER_ANTERIOR = False -> se conserva el historico.")

## Paso 6 — Refrescar el servicio sin downtime (endpoint corregido)

La guía original apuntaba a un endpoint de admin
(`/admin/system/publishingtools/refreshservice`) que no corresponde al
operador real. `RefreshService` es una tarea del servicio de
geoprocesamiento out-of-the-box `PublishingTools`, y requiere que el
servicio tenga `hasLiveData: true` (activado una vez desde Server
Manager — paso 5 de `01_setup_inicial_viento.ipynb`). `RUTA_HOST_AGS`
es un archivo de conexión de publicador/administrador a tu ArcGIS
Server (el mismo tipo de conexión `.ags` que usarías en el panel de
Catálogo de ArcGIS Pro).

In [ ]:
arcpy.ImportToolbox(rf"{RUTA_HOST_AGS};System/PublishingTools")
arcpy.PublishingTools.RefreshService(NOMBRE_SERVICIO, "ImageServer", CARPETA_SERVICIO, "#")
print("RefreshService ejecutado.")

## Paso 7 — Validar que el servicio quedó actualizado

In [ ]:
resp = requests.get(IMAGE_SERVICE_URL, params={"f": "json"}, timeout=30)
resp.raise_for_status()
info = resp.json()

print(f"Servicio: {info.get('name', NOMBRE_SERVICIO)}")
print(f"Extension: {info.get('extent')}")
print(f"Bandas: {info.get('bandCount')}")
print("Confirma en el visor web que el streamline refleja el periodo mas reciente.")

## Cómo programar este notebook

**Opción recomendada — ArcGIS Notebook Server (dentro de Enterprise):**
1. Sube este notebook como ítem de Portal (o ábrelo directo en Notebook Server).
2. En el panel de notebooks, usa **Manage Tasks / Programar tarea** para
   definir la frecuencia (mensual, semanal, etc.) y la hora de ejecución.
3. Corre **Run all cells** manualmente antes de dejarlo desatendido, para
   confirmar permisos de red hacia la carpeta registrada y hacia el
   servicio de geoprocesamiento `PublishingTools`.

**Alternativa — ArcGIS Pro + Task Scheduler (si aún no tienes Notebook Server):**
Exporta este notebook a `.py`:

```
jupyter nbconvert --to script 02_actualizacion_periodica_viento.ipynb
```

Y prográmalo igual que en la guía Enterprise original, apuntando al
`python.exe` del entorno clonado de Pro.

Ante cualquier falla, revisa la sección 13 (Errores comunes) del README.